# 02. Бэктест моделей прогноза расходов МО

**Схема.** Rolling-origin с расширяющимся окном. Точка отсечки `T` — конец месяца, на который делается
прогноз; модель видит только данные, известные к `T`, и прогнозирует расходы в месяце `T+h`.
Для `h = 1, 3, 6` берём все отсечки с января 2024, для которых `T+h ≤ декабрь 2024`.
Для `h = 12` истории хватает на одно окно: прогноз с декабря 2023 на декабрь 2024.

**Модели.**
- `naive` — расходы через `h` месяцев равны текущим;
- `seasonal_naive` — равны расходам в том же месяце год назад (если его нет — как `naive`);
- `prophet` — базовая модель организаторов, обучается отдельно на каждый МО;
- `seasonal_naive_trend` — прошлогоднее значение с поправкой на рост за год;
- `lgbm_by_h` — глобальная LightGBM на все МО, отдельная на каждый горизонт; `lgbm_by_h_noreg` — то же без категории «регион»;
- `lgbm_pooled` — одна LightGBM на горизонты 1–6, горизонт — признак (больше обучающих пар для дальних горизонтов);
- `ensemble` — среднее Prophet и `lgbm_pooled`; `ens_snt_lgbm` — среднее `seasonal_naive_trend` и `lgbm_pooled`.

**Метрики.** MAE (обязательная) и R². Считаются по всем парам «МО × отсечка» горизонта.
Prophet обучается на выборке МО (см. конфиг), поэтому главная таблица сравнения — на этой общей выборке;
для naive и LightGBM дополнительно приведены метрики на всех МО.

In [ ]:
import logging, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "configs" / "backtest.yaml", encoding="utf-8"))
OUT = ROOT / "results"; OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(CFG["seed"])

panel = pd.read_parquet(ROOT / CFG["data"]["panel"])
panel = panel[panel.date >= "2023-01-01"].copy()
Y = CFG["data"]["target"]
H = CFG["horizons"]
print(panel.shape, panel.mo_id.nunique(), "МО")

## 1. Точки отсечки и признаки

In [ ]:
first, last = pd.Timestamp(CFG["backtest"]["first_origin"]), pd.Timestamp(CFG["backtest"]["last_target"])
def origins(h):
    if h == 12: return [pd.Timestamp(CFG["backtest"]["h12_origin"])]
    return list(pd.date_range(first, last - pd.DateOffset(months=h), freq="MS"))
ORIG = {h: origins(h) for h in H}
for h in H: print(f"h={h}: отсечек {len(ORIG[h])}, {ORIG[h][0].date()} … {ORIG[h][-1].date()}")

drop = {"mo_id", "date", "mo_name_sber", "mo_name_rosstat", "region", "has_target_now"}
FEATS = [c for c in panel.columns if c not in drop and not c.startswith(("target_", "target_date"))]
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(panel[c])]
for c in CATS: panel[c] = panel[c].astype("category")
print(f"признаков: {len(FEATS)}, категориальных: {CATS}")

# выборка МО для Prophet: стратифицированно по классу размера
mo_info = panel.drop_duplicates("mo_id")[["mo_id", "size_class"]]
n = CFG["models"]["prophet"]["sample_n_mo"]
SAMPLE = (mo_info.groupby("size_class", observed=True, group_keys=False)
          .apply(lambda d: d.sample(max(1, round(n * len(d) / len(mo_info))), random_state=CFG["seed"]))
          .mo_id.tolist())
print("МО в выборке для Prophet:", len(SAMPLE))

## 2. Базовые модели: naive и seasonal naive

In [ ]:
yy = panel.pivot(index="mo_id", columns="date", values=Y)
def baseline_preds(h):
    rows = []
    for T in ORIG[h]:
        tgt = T + pd.DateOffset(months=h)
        if tgt not in yy.columns: continue
        same_month_ly = tgt - pd.DateOffset(months=12)
        d = pd.DataFrame({"mo_id": yy.index, "origin": T, "h": h, "y_true": yy[tgt].values,
                          "naive": yy[T].values})
        d["seasonal_naive"] = yy[same_month_ly].values if same_month_ly in yy.columns else np.nan
        d["seasonal_naive"] = d.seasonal_naive.fillna(d.naive)
        # с поправкой на тренд: прошлогоднее значение × рост с прошлого года до текущего месяца
        ly_T = T - pd.DateOffset(months=12)
        if same_month_ly in yy.columns and ly_T in yy.columns:
            d["seasonal_naive_trend"] = yy[same_month_ly].values * yy[T].values / yy[ly_T].values
        else:
            d["seasonal_naive_trend"] = np.nan
        d["seasonal_naive_trend"] = d.seasonal_naive_trend.fillna(d.naive)
        rows.append(d)
    return pd.concat(rows)
preds = {h: baseline_preds(h) for h in H}

## 3. Prophet (базовая модель организаторов)

In [ ]:
from prophet import Prophet
cutoffs = sorted({T for h in H for T in ORIG[h]})
P = CFG["models"]["prophet"]["params"]
cache = OUT / f"prophet_forecasts_n{len(SAMPLE)}.parquet"      # Prophet долгий — кешируем
t0, rows = time.time(), []
for mo in ([] if cache.exists() else SAMPLE):
    s = yy.loc[mo].dropna()
    for T in cutoffs:
        hist = s[s.index <= T]
        if len(hist) < 6: continue
        m = Prophet(**P)
        m.fit(pd.DataFrame({"ds": hist.index, "y": hist.values}))
        fut = pd.DataFrame({"ds": pd.date_range(T + pd.DateOffset(months=1), periods=12, freq="MS")})
        fc = m.predict(fut)[["ds", "yhat"]]
        fc["mo_id"], fc["origin"] = mo, T
        rows.append(fc)
prophet_fc = pd.read_parquet(cache) if cache.exists() else pd.concat(rows)
prophet_fc.to_parquet(cache, index=False)
print(f"Prophet: {len(SAMPLE)} МО × {len(cutoffs)} отсечек за {time.time() - t0:.0f} c")
for h in H:
    p = prophet_fc.assign(h=((prophet_fc.ds.dt.year - prophet_fc.origin.dt.year) * 12
                              + prophet_fc.ds.dt.month - prophet_fc.origin.dt.month))
    p = p[p.h == h][["mo_id", "origin", "yhat"]].rename(columns={"yhat": "prophet"})
    preds[h] = preds[h].merge(p, on=["mo_id", "origin"], how="left")

## 4. LightGBM: глобальная модель, отдельная на каждый горизонт
Цель — `log(y_{T+h} / y_T)`: модель учит относительное изменение, поэтому большие и маленькие МО
сравнимы. Обучаемся только на парах, у которых `T+h` уже наступил к моменту отсечки (никакой утечки).

In [ ]:
LP = CFG["models"]["lightgbm"]["params"]
POOL_H = CFG["models"]["lightgbm"]["pooled_horizons"]
importances = []
base = panel.dropna(subset=[Y]).copy()

def make_pairs(hs):
    """Пары (строка t, горизонт h) с целью log(y_{t+h}/y_t); горизонт — признак."""
    parts = []
    for h in hs:
        d = base.copy(); d["h"] = h
        d["z"] = np.log(d[f"target_h{h}"] / d[Y]) if f"target_h{h}" in d else np.nan
        d["target_date"] = d.date + pd.DateOffset(months=h)
        parts.append(d)
    return pd.concat(parts)

def run_lgbm(name, feats, pooled):
    cats = [c for c in feats if c in CATS]
    hs_train = POOL_H if pooled else None
    for h in H:
        if pooled and h not in POOL_H: continue
        pairs = make_pairs(hs_train if pooled else [h])
        pairs = pairs[pairs.z.notna() & np.isfinite(pairs.z)]
        X = feats + (["h"] if pooled else [])
        out = []
        for T in ORIG[h]:
            train = pairs[pairs.target_date <= T]             # цель уже наступила к моменту отсечки
            if len(train) < 1000: continue
            test = base[base.date == T].assign(h=h)
            model = lgb.LGBMRegressor(**LP, random_state=CFG["seed"])
            model.fit(train[X], train.z, categorical_feature=cats)
            out.append(pd.DataFrame({"mo_id": test.mo_id.values, "origin": T,
                                     name: test[Y].values * np.exp(model.predict(test[X]))}))
            if name == "lgbm_pooled":
                importances.append(pd.Series(model.booster_.feature_importance("gain"), index=X))
        if out: preds[h] = preds[h].merge(pd.concat(out), on=["mo_id", "origin"], how="left")
        print(f"{name}, h={h}: моделей {len(out)}")

FEATS_NOREG = [c for c in FEATS if c != "region_key"]
t0 = time.time()
run_lgbm("lgbm_by_h", FEATS, pooled=False)            # отдельная модель на каждый горизонт
run_lgbm("lgbm_by_h_noreg", FEATS_NOREG, pooled=False) # абляция: без категории «регион»
run_lgbm("lgbm_pooled", FEATS_NOREG, pooled=True)       # общая модель на горизонты 1–6
print(f"LightGBM: {time.time() - t0:.0f} c")

# ансамбль: простое среднее Prophet и общей LightGBM (веса не подбираем на тесте)
for h in H:
    if "lgbm_pooled" in preds[h]:
        preds[h]["ensemble"] = preds[h][["prophet", "lgbm_pooled"]].mean(axis=1, skipna=False)
        # лучший вариант: сезонность с трендом ловит декабрьский пик, LightGBM — всё остальное
        preds[h]["ens_snt_lgbm"] = preds[h][["seasonal_naive_trend", "lgbm_pooled"]].mean(axis=1, skipna=False)

## 5. Метрики

In [ ]:
MODELS = ["naive", "seasonal_naive", "seasonal_naive_trend", "prophet", "lgbm_by_h", "lgbm_by_h_noreg", "lgbm_pooled", "ensemble", "ens_snt_lgbm"]
def metrics(df, subset_name):
    res = []
    for h in H:
        d = df[h].dropna(subset=["y_true"])
        if subset_name == "выборка Prophet": d = d[d.mo_id.isin(SAMPLE)]
        for m in MODELS:
            if m not in d or d[m].isna().all(): continue
            x = d.dropna(subset=[m])
            res.append({"набор МО": subset_name, "h": h, "модель": m, "n": len(x),
                        "MAE": mean_absolute_error(x.y_true, x[m]), "R2": r2_score(x.y_true, x[m]),
                        "MAPE_%": np.mean(np.abs(x[m] / x.y_true - 1)) * 100})
    return pd.DataFrame(res)

table = pd.concat([metrics(preds, "выборка Prophet"), metrics(preds, "все МО")])
table.to_csv(OUT / "forecast_metrics.csv", index=False)
pivot = table.pivot_table(index=["набор МО", "модель"], columns="h", values="MAE").round(0)
print("MAE, руб. (меньше — лучше)\n", pivot.to_string())
print("\nR²\n", table.pivot_table(index=["набор МО", "модель"], columns="h", values="R2").round(3).to_string())

## 6. Стабильность по отсечкам и важность признаков

In [ ]:
by_origin = []
for h in [1, 3, 6]:
    d = preds[h][preds[h].mo_id.isin(SAMPLE)].dropna(subset=["y_true"])
    for T, g in d.groupby("origin"):
        row = {"h": h, "origin": T.date()}
        for m in MODELS:
            if m in g and g[m].notna().any():
                x = g.dropna(subset=[m]); row[m] = mean_absolute_error(x.y_true, x[m])
        by_origin.append(row)
by_origin = pd.DataFrame(by_origin).round(0); by_origin.to_csv(OUT / "mae_by_origin.csv", index=False)
print(by_origin.to_string(index=False))

imp = pd.concat(importances, axis=1).mean(axis=1).sort_values(ascending=False)
(imp / imp.sum()).head(20).round(3).to_csv(OUT / "lgbm_feature_importance_top20.csv")
print("\nТоп-15 признаков LightGBM (доля gain):\n", (imp / imp.sum()).head(15).round(3).to_string())

for h in H: preds[h].to_parquet(OUT / f"predictions_h{h}.parquet", index=False)